# Bias in Bios Leakage Experiment

This notebook evaluates target-conditioned adversarial regularization on the original 28-way profession task. The actor and classifier are trained end-to-end from hard Concept-QA responses using a fixed actor temperature. Sequential evaluation and query-composition statistics are aggregated over five random seeds.


In [1]:
import copy
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm


def find_repo_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "assets" / "concepts" / "bias_in_bios.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root from current working directory")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from claq.core import (
    file_sha256,
    load_answer_cache,
    load_run_bundle,
    save_answer_cache,
    save_bundle_checkpoint,
)
from claq.models import ConceptAnswererMLP, Network
from claq.training import GradientReversal, HistorySamplingConfig, sample_history_mask, seed_everything

EXPERIMENT = "bias_in_bios_leakage"
SEEDS = (0, 1, 2, 3, 4)
seed_everything(SEEDS[0])

sample_dir = repo_root / "artifacts" / "data" / "bias_in_bios_sample"
concepts_path = repo_root / "assets" / "concepts" / "bias_in_bios.csv"
labels_wide_path = sample_dir / "labels_wide_test_train_validation.csv"
qa_checkpoint_path = repo_root / "artifacts" / "models" / "concept_qa_bias_in_bios_concept_answerer_mlp_openai_labels.pt"
models_dir = repo_root / "artifacts" / "models"
runs_dir = repo_root / "artifacts" / "runs"
answer_cache_dir = repo_root / "artifacts" / "concept_answers" / "bias_in_bios"
models_dir.mkdir(parents=True, exist_ok=True)
runs_dir.mkdir(parents=True, exist_ok=True)
answer_cache_dir.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "validation", "test"]
TEXT_COLUMN = "hard_text"
MAX_QUESTIONS = 20
NUM_EPOCHS = 50
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
ACTOR_EPS = 1.0
TRAIN_MIN_HISTORY = 0
TRAIN_MAX_HISTORY = 20
CONFIDENCE_THRESHOLDS = [0.50, 0.60, 0.70, 0.80, 0.90]
FORCE_RETRAIN = False
FORCE_REBUILD_ANSWER_CACHE = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

{
    "repo_root": str(repo_root),
    "sample_dir": str(sample_dir),
    "labels_wide_path": str(labels_wide_path),
    "qa_checkpoint_path": str(qa_checkpoint_path),
    "device": str(device),
}


{'repo_root': '/home/jupyter/claq',
 'sample_dir': '/home/jupyter/claq/artifacts/data/bias_in_bios_sample',
 'labels_wide_path': '/home/jupyter/claq/artifacts/data/bias_in_bios_sample/labels_wide_test_train_validation.csv',
 'qa_checkpoint_path': '/home/jupyter/claq/artifacts/models/concept_qa_bias_in_bios_concept_answerer_mlp_openai_labels.pt',
 'device': 'cuda'}

In [2]:
concepts_df = pd.read_csv(concepts_path)
concept_names = concepts_df["concept"].tolist()
labels_wide = pd.read_csv(labels_wide_path)

split_frames = {}
for split in SPLITS:
    samples = pd.read_csv(sample_dir / f"{split}.csv").reset_index(names="sample_row")
    labels = labels_wide.loc[labels_wide["split"].eq(split)].copy()
    merged = samples.merge(
        labels[["sample_row", "source_index", *concept_names]],
        on=["sample_row", "source_index"],
        how="inner",
        validate="one_to_one",
    )
    if len(merged) != len(samples):
        raise ValueError(f"Merged {len(merged)} rows for {split}, expected {len(samples)}")
    split_frames[split] = merged

profession_id_to_name = (
    split_frames["train"][["profession", "profession_name"]]
    .drop_duplicates()
    .sort_values("profession")
    .set_index("profession")["profession_name"]
    .to_dict()
)
num_professions = len(profession_id_to_name)

oracle_labels_01 = {
    split: frame[concept_names].to_numpy(dtype=np.float32)
    for split, frame in split_frames.items()
}
oracle_answers = {
    split: np.where(labels > 0, 1.0, -1.0).astype(np.float32)
    for split, labels in oracle_labels_01.items()
}
y_by_split = {
    split: frame["profession"].to_numpy(dtype=int)
    for split, frame in split_frames.items()
}
s_by_split = {
    split: frame["gender"].to_numpy(dtype=np.float32)
    for split, frame in split_frames.items()
}

{
    "split_rows": {split: len(frame) for split, frame in split_frames.items()},
    "num_professions": num_professions,
    "num_concepts": len(concept_names),
    "sensitive_target": "dataset gender label, 1=female and 0=male",
    "female_share_by_split": {split: float(values.mean()) for split, values in s_by_split.items()},
    "max_questions": MAX_QUESTIONS,
}


{'split_rows': {'train': 10000, 'validation': 1000, 'test': 2000},
 'num_professions': 28,
 'num_concepts': 58,
 'sensitive_target': 'dataset gender label, 1=female and 0=male',
 'female_share_by_split': {'train': 0.44859999418258667,
  'validation': 0.44699999690055847,
  'test': 0.44699999690055847},
 'max_questions': 20}

In [3]:
answer_cache_paths = {
    split: answer_cache_dir / f"{split}_hard_answers.pt" for split in SPLITS
}
answer_cache_metadata = {
    "experiment": "bias_in_bios",
    "concept_count": len(concept_names),
    "qa_checkpoint": qa_checkpoint_path.name,
    "qa_checkpoint_sha256": file_sha256(qa_checkpoint_path),
}
use_cached_answers = (
    not FORCE_REBUILD_ANSWER_CACHE
    and all(path.exists() for path in answer_cache_paths.values())
)

if use_cached_answers:
    answer_cache_payloads = {
        split: load_answer_cache(path, expected_metadata=answer_cache_metadata)
        for split, path in answer_cache_paths.items()
    }
    qa_answers = {
        split: payload["answers"].numpy().astype(np.float32)
        for split, payload in answer_cache_payloads.items()
    }
    decision_threshold = 0.5
else:
    checkpoint = torch.load(qa_checkpoint_path, map_location=device, weights_only=False)
    encoder_name = checkpoint["encoder_name"]
    model_config = checkpoint["model_config"]
    model_config["hidden_dims"] = tuple(model_config["hidden_dims"])
    concept_embeddings = checkpoint["concept_embeddings"].to(device).float()
    decision_threshold = float(checkpoint.get("decision_threshold", 0.5))

    concept_answerer = ConceptAnswererMLP(**model_config).to(device)
    concept_answerer.load_state_dict(checkpoint["model_state_dict"])
    concept_answerer.eval()
    encoder = SentenceTransformer(encoder_name, device=str(device))
    text_embeddings = {
        split: encoder.encode(
            frame[TEXT_COLUMN].fillna("").tolist(),
            batch_size=128,
            show_progress_bar=True,
            convert_to_numpy=True,
        ).astype(np.float32)
        for split, frame in split_frames.items()
    }

{
    "answer_source": "cache" if use_cached_answers else "Concept-QA inference",
    "cache_paths": {split: str(path) for split, path in answer_cache_paths.items()},
    "decision_threshold": decision_threshold,
}


{'answer_source': 'cache',
 'cache_paths': {'train': '/home/jupyter/claq/artifacts/concept_answers/bias_in_bios/train_hard_answers.pt',
  'validation': '/home/jupyter/claq/artifacts/concept_answers/bias_in_bios/validation_hard_answers.pt',
  'test': '/home/jupyter/claq/artifacts/concept_answers/bias_in_bios/test_hard_answers.pt'},
 'decision_threshold': 0.5}

In [4]:
def build_text_concept_inputs(text_batch, concept_embeddings):
    repeated_text = text_batch.repeat_interleave(len(concept_names), dim=0)
    repeated_concepts = concept_embeddings.repeat(text_batch.size(0), 1)
    return torch.cat([repeated_text, repeated_concepts], dim=1)


@torch.no_grad()
def predict_concept_scores(text_embedding_array, batch_size=256):
    concept_answerer.eval()
    scores = []
    text_tensor = torch.tensor(text_embedding_array, dtype=torch.float32, device=device)

    for start in tqdm(range(0, len(text_tensor), batch_size), desc="Predict Concept-QA scores"):
        text_batch = text_tensor[start : start + batch_size]
        inputs = build_text_concept_inputs(text_batch, concept_embeddings)
        logits = concept_answerer(inputs).view(text_batch.size(0), len(concept_names))
        scores.append(torch.sigmoid(logits).cpu().numpy())

    return np.concatenate(scores, axis=0).astype(np.float32)


if not use_cached_answers:
    qa_scores = {split: predict_concept_scores(text_embeddings[split]) for split in SPLITS}
    qa_labels_01 = {
        split: (scores >= decision_threshold).astype(np.float32)
        for split, scores in qa_scores.items()
    }
    qa_answers = {
        split: np.where(labels > 0, 1.0, -1.0).astype(np.float32)
        for split, labels in qa_labels_01.items()
    }
    answer_cache_payloads = {}
    for split in SPLITS:
        save_answer_cache(
            answer_cache_paths[split],
            answers=torch.from_numpy(qa_answers[split]),
            labels=torch.from_numpy(y_by_split[split]),
            sensitive_targets=torch.from_numpy(s_by_split[split]),
            metadata=answer_cache_metadata,
        )
        answer_cache_payloads[split] = load_answer_cache(
            answer_cache_paths[split], expected_metadata=answer_cache_metadata
        )
else:
    qa_labels_01 = {
        split: (answers > 0).astype(np.float32)
        for split, answers in qa_answers.items()
    }

{
    "decision_threshold": decision_threshold,
    "answer_source": "cache" if use_cached_answers else "computed and cached",
    "qa_positive_rate": {split: float(labels.mean()) for split, labels in qa_labels_01.items()},
    "oracle_positive_rate": {split: float(labels.mean()) for split, labels in oracle_labels_01.items()},
}


{'decision_threshold': 0.5,
 'answer_source': 'cache',
 'qa_positive_rate': {'train': 0.08086379617452621,
  'validation': 0.07868965715169907,
  'test': 0.08043103665113449},
 'oracle_positive_rate': {'train': 0.08865000307559967,
  'validation': 0.08668965846300125,
  'test': 0.08932758867740631}}

In [5]:
def make_answer_dataset(answer_matrix, labels, sensitive_labels):
    return TensorDataset(
        torch.tensor(answer_matrix, dtype=torch.float32),
        torch.tensor(labels, dtype=torch.long),
        torch.tensor(sensitive_labels, dtype=torch.float32),
    )


train_loader = DataLoader(
    make_answer_dataset(qa_answers["train"], y_by_split["train"], s_by_split["train"]),
    batch_size=BATCH_SIZE,
    shuffle=True,
)
validation_loader = DataLoader(
    make_answer_dataset(qa_answers["validation"], y_by_split["validation"], s_by_split["validation"]),
    batch_size=BATCH_SIZE,
    shuffle=False,
)
test_loader = DataLoader(
    make_answer_dataset(qa_answers["test"], y_by_split["test"], s_by_split["test"]),
    batch_size=BATCH_SIZE,
    shuffle=False,
)

sensitive_indices = torch.tensor(
    concepts_df.index[concepts_df["kind"].ne("utility")].tolist(),
    dtype=torch.long,
    device=device,
)
history_config = HistorySamplingConfig(
    min_history=TRAIN_MIN_HISTORY,
    max_history=TRAIN_MAX_HISTORY,
    non_sensitive_only=False,
)


def build_neural_claq_models(actor_eps=ACTOR_EPS):
    actor = Network(query_size=len(concept_names), output_size=len(concept_names), eps=actor_eps).to(device)
    classifier = Network(query_size=len(concept_names), output_size=num_professions, eps=None).to(device)
    sensitive_head = Network(
        query_size=len(concept_names) + num_professions,
        output_size=1,
        eps=None,
    ).to(device)
    return actor, classifier, sensitive_head

{
    "train_batches": len(train_loader),
    "validation_batches": len(validation_loader),
    "test_batches": len(test_loader),
    "history_config": history_config,
    "sensitive_target": "dataset gender label, 1=female and 0=male",
    "sensitive_conditioning": "encoded knowledge state and one-hot profession label",
    "female_share_by_split": {split: float(values.mean()) for split, values in s_by_split.items()},
    "sensitive_concepts": concepts_df.loc[sensitive_indices.cpu().numpy(), "concept"].tolist(),
}


{'train_batches': 40,
 'validation_batches': 4,
 'test_batches': 8,
 'history_config': HistorySamplingConfig(min_history=0, max_history=20, non_sensitive_only=False),
 'sensitive_target': 'dataset gender label, 1=female and 0=male',
 'sensitive_conditioning': 'encoded knowledge state and one-hot profession label',
 'female_share_by_split': {'train': 0.44859999418258667,
  'validation': 0.44699999690055847,
  'test': 0.44699999690055847},
 'sensitive_concepts': ['male_pronouns',
  'female_pronouns',
  'male_gendered_titles',
  'female_gendered_titles',
  'spouse_or_partner_mentions',
  'parent_role_mentions',
  'child_or_family_care_mentions',
  'appearance_or_body_presentation',
  'gendered_organization_or_award']}

In [6]:
def batch_metrics_from_logits(logits, labels):
    predictions = logits.argmax(dim=1)
    correct = int((predictions == labels).sum().item())
    return correct, int(labels.numel())


def run_neural_claq_epoch(
    loader,
    actor,
    classifier,
    sensitive_head,
    optimizer=None,
    lambda_s=0.0,
    train=True,
):
    actor.train(train)
    classifier.train(train)
    sensitive_head.train(train)

    total = 0
    correct = 0
    loss_total = 0.0
    task_total = 0.0
    sens_total = 0.0
    q_entropy_total = 0.0
    sensitive_query_total = 0.0
    num_batches = 0

    for answers, labels, sensitive_target in tqdm(loader, desc="Train" if train else "Evaluate", leave=False):
        answers = answers.to(device)
        labels = labels.to(device)
        sensitive_target = sensitive_target.to(device)

        with torch.set_grad_enabled(train):
            mask, masked_answers = sample_history_mask(
                answers=answers,
                config=history_config,
                sensitive_indices=sensitive_indices,
            )
            query_distribution = actor(masked_answers, mask)
            updated_answers = masked_answers + query_distribution * answers

            logits = classifier(updated_answers)
            loss_task = F.cross_entropy(logits, labels)

            reversed_knowledge_state = GradientReversal.apply(updated_answers, lambda_s)
            profession_labels = F.one_hot(labels, num_classes=num_professions).to(
                device=device,
                dtype=updated_answers.dtype,
            )
            sensitive_input = torch.cat((reversed_knowledge_state, profession_labels), dim=1)
            sensitive_logits = sensitive_head(sensitive_input).squeeze(-1)
            loss_sensitive = F.binary_cross_entropy_with_logits(sensitive_logits, sensitive_target)

            if sensitive_indices.numel() > 0:
                sensitive_query_rate = query_distribution[:, sensitive_indices].sum(dim=1).mean()
            else:
                sensitive_query_rate = torch.zeros((), device=device)

            loss = loss_task + loss_sensitive

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        batch_correct, batch_total = batch_metrics_from_logits(logits.detach(), labels)
        correct += batch_correct
        total += batch_total
        loss_total += float(loss.detach().item())
        task_total += float(loss_task.detach().item())
        sens_total += float(loss_sensitive.detach().item())
        sensitive_query_total += float(sensitive_query_rate.detach().item())
        q_safe = query_distribution.detach().clamp_min(1e-8)
        q_entropy_total += float((-(q_safe * torch.log(q_safe)).sum(dim=1)).mean().item())
        num_batches += 1

    return {
        "accuracy": correct / max(total, 1),
        "loss": loss_total / max(num_batches, 1),
        "task_loss": task_total / max(num_batches, 1),
        "sensitive_loss": sens_total / max(num_batches, 1),
        "sensitive_query_rate": sensitive_query_total / max(num_batches, 1),
        "query_entropy": q_entropy_total / max(num_batches, 1),
    }


def fit_neural_claq(
    run_name,
    lambda_s=0.0,
    num_epochs=NUM_EPOCHS,
    seed=0,
):
    seed_everything(seed)
    actor, classifier, sensitive_head = build_neural_claq_models(actor_eps=ACTOR_EPS)
    optimizer = torch.optim.Adam(
        list(actor.parameters()) + list(classifier.parameters()) + list(sensitive_head.parameters()),
        lr=LEARNING_RATE,
    )

    history = []
    best = {"validation_accuracy": -1.0}
    for epoch in tqdm(range(1, num_epochs + 1), desc=f"{run_name} seed={seed}"):
        train_metrics = run_neural_claq_epoch(
            train_loader,
            actor,
            classifier,
            sensitive_head,
            optimizer=optimizer,
            lambda_s=lambda_s,
            train=True,
        )
        validation_metrics = run_neural_claq_epoch(
            validation_loader,
            actor,
            classifier,
            sensitive_head,
            lambda_s=lambda_s,
            train=False,
        )
        row = {
            "run_name": run_name,
            "seed": seed,
            "epoch": epoch,
            "lambda_s": lambda_s,
            "actor_eps": actor.eps,
            **{f"train_{key}": value for key, value in train_metrics.items()},
            **{f"validation_{key}": value for key, value in validation_metrics.items()},
        }
        history.append(row)

        if validation_metrics["accuracy"] >= best["validation_accuracy"]:
            best = {
                "validation_accuracy": validation_metrics["accuracy"],
                "epoch": epoch,
                "actor_state_dict": copy.deepcopy(actor.state_dict()),
                "classifier_state_dict": copy.deepcopy(classifier.state_dict()),
                "sensitive_head_state_dict": copy.deepcopy(sensitive_head.state_dict()),
                "history_row": row,
            }

    actor.load_state_dict(best["actor_state_dict"])
    classifier.load_state_dict(best["classifier_state_dict"])
    sensitive_head.load_state_dict(best["sensitive_head_state_dict"])
    return {
        "run_name": run_name,
        "seed": seed,
        "lambda_s": lambda_s,
        "history": pd.DataFrame(history),
        "best": best,
        "actor": actor,
        "classifier": classifier,
        "sensitive_head": sensitive_head,
    }


def load_or_train_neural_claq(run_name, lambda_s, seed, force_retrain=FORCE_RETRAIN):
    run_stem = f"{EXPERIMENT}_{run_name}_seed_{seed}"
    checkpoint_path = models_dir / f"{run_stem}_best.pt"
    history_path = runs_dir / f"{run_stem}_history.csv"

    if checkpoint_path.exists() and history_path.exists() and not force_retrain:
        bundle = load_run_bundle(
            checkpoint_path,
            device=device,
            max_queries=len(concept_names),
            num_classes=num_professions,
            actor_eps=ACTOR_EPS,
        )
        meta = bundle["meta"]
        return {
            "run_name": run_name,
            "seed": seed,
            "lambda_s": lambda_s,
            "history": pd.read_csv(history_path),
            "best": {
                "validation_accuracy": meta["best_validation_accuracy"],
                "epoch": meta["best_epoch"],
                "history_row": meta.get("best_history_row", {}),
            },
            "actor": bundle["actor"],
            "classifier": bundle["classifier"],
            "sensitive_head": bundle["s_head"],
            "checkpoint_path": checkpoint_path,
        }

    run = fit_neural_claq(run_name=run_name, lambda_s=lambda_s, seed=seed)
    best = run["best"]
    save_bundle_checkpoint(
        checkpoint_path,
        actor=run["actor"],
        classifier=run["classifier"],
        s_head=run["sensitive_head"],
        metadata={
            "experiment_name": EXPERIMENT,
            "run_name": run_name,
            "seed": seed,
            "lambda_s": lambda_s,
            "actor_eps": ACTOR_EPS,
            "max_queries": len(concept_names),
            "num_classes": num_professions,
            "best_validation_accuracy": best["validation_accuracy"],
            "best_epoch": best["epoch"],
            "best_history_row": best["history_row"],
            "sensitive_conditioning": "conditional_y",
        },
    )
    run["history"].to_csv(history_path, index=False, lineterminator="\n")
    run["best"] = {
        "validation_accuracy": best["validation_accuracy"],
        "epoch": best["epoch"],
        "history_row": best["history_row"],
    }
    run["checkpoint_path"] = checkpoint_path
    return run


In [7]:
@torch.no_grad()
def rollout_neural_claq(actor, classifier, loader, run_name, seed, max_questions=MAX_QUESTIONS):
    actor.eval()
    classifier.eval()
    curve_rows = []
    detail_rows = []
    query_rows = []
    row_offset = 0

    for answers, labels, _sensitive_target in tqdm(loader, desc=f"Rollout {run_name}"):
        answers = answers.to(device)
        labels = labels.to(device)
        batch_size = answers.size(0)
        mask = torch.zeros_like(answers)
        masked_answers = torch.zeros_like(answers)
        row_indices = np.arange(row_offset, row_offset + batch_size)

        for budget in range(max_questions + 1):
            logits = classifier(masked_answers)
            probabilities = F.softmax(logits, dim=1)
            confidence, predictions = probabilities.max(dim=1)

            detail_rows.append(
                pd.DataFrame(
                    {
                        "run_name": run_name,
                        "seed": seed,
                        "budget": budget,
                        "row_idx": row_indices,
                        "label": labels.cpu().numpy(),
                        "prediction": predictions.cpu().numpy(),
                        "confidence": confidence.cpu().numpy(),
                        "correct": (predictions == labels).cpu().numpy(),
                    }
                )
            )

            if budget == max_questions:
                break

            query_distribution = actor(masked_answers, mask)
            chosen_queries = query_distribution.argmax(dim=1)
            query_rows.append(
                pd.DataFrame(
                    {
                        "run_name": run_name,
                        "seed": seed,
                        "budget": budget + 1,
                        "row_idx": row_indices,
                        "query_index": chosen_queries.cpu().numpy(),
                        "query_concept": [concept_names[idx] for idx in chosen_queries.cpu().numpy()],
                    }
                )
            )
            masked_answers = masked_answers + query_distribution * answers
            mask = torch.clamp(mask + query_distribution, 0.0, 1.0)

        row_offset += batch_size

    detail_df = pd.concat(detail_rows, ignore_index=True)
    query_df = pd.concat(query_rows, ignore_index=True)

    for budget, budget_detail in detail_df.groupby("budget"):
        curve_rows.append(
            {
                "run_name": run_name,
                "seed": seed,
                "budget": int(budget),
                "accuracy": accuracy_score(budget_detail["label"], budget_detail["prediction"]),
                "macro_f1": f1_score(budget_detail["label"], budget_detail["prediction"], average="macro"),
                "mean_confidence": float(budget_detail["confidence"].mean()),
            }
        )

    return pd.DataFrame(curve_rows).sort_values("budget"), detail_df, query_df


def summarize_confidence_stopping(detail_df, thresholds=CONFIDENCE_THRESHOLDS):
    rows = []
    run_name = detail_df["run_name"].iloc[0]
    seed = int(detail_df["seed"].iloc[0])
    for threshold in thresholds:
        reached = detail_df.loc[detail_df["confidence"].ge(threshold)].copy()
        if reached.empty:
            rows.append(
                {
                    "run_name": run_name,
                    "seed": seed,
                    "confidence_threshold": threshold,
                    "coverage": 0.0,
                    "mean_questions_when_reached": np.nan,
                    "accuracy_when_reached": np.nan,
                }
            )
            continue

        first_reached = reached.sort_values("budget").drop_duplicates("row_idx", keep="first")
        rows.append(
            {
                "run_name": run_name,
                "seed": seed,
                "confidence_threshold": threshold,
                "coverage": len(first_reached) / detail_df["row_idx"].nunique(),
                "mean_questions_when_reached": first_reached["budget"].mean(),
                "accuracy_when_reached": first_reached["correct"].mean(),
            }
        )
    return pd.DataFrame(rows)


def summarize_query_usage(query_df):
    return (
        query_df.groupby(["run_name", "seed", "budget", "query_concept"])
        .size()
        .reset_index(name="count")
        .sort_values(["run_name", "seed", "budget", "count"], ascending=[True, True, True, False])
    )


In [8]:
runs_by_seed = {}
for seed in SEEDS:
    runs_by_seed[seed] = {
        "baseline": load_or_train_neural_claq(
            run_name="neural_baseline_lambda_s_0",
            lambda_s=0.0,
            seed=seed,
        ),
        "claq": load_or_train_neural_claq(
            run_name="neural_claq_lambda_s_0_4",
            lambda_s=0.4,
            seed=seed,
        ),
    }

trained_runs = [run for seed_runs in runs_by_seed.values() for run in seed_runs.values()]
history_df = pd.concat([run["history"] for run in trained_runs], ignore_index=True)

run_summary = pd.DataFrame(
    [
        {
            "run_name": run["run_name"],
            "seed": run["seed"],
            "best_epoch": run["best"]["epoch"],
            "validation_accuracy": run["best"]["validation_accuracy"],
        }
        for run in trained_runs
    ]
)
run_summary.groupby("run_name").agg(
    seeds=("seed", "nunique"),
    validation_accuracy_mean=("validation_accuracy", "mean"),
    validation_accuracy_std=("validation_accuracy", "std"),
).reset_index()


neural_baseline_lambda_s_0 seed=0:   0%|          | 0/50 [00:00<?, ?it/s]

Train:   0%|          | 0/40 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/4 [00:00<?, ?it/s]

Train:   0%|          | 0/40 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/4 [00:00<?, ?it/s]

...

Train:   0%|          | 0/40 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/4 [00:00<?, ?it/s]

Train:   0%|          | 0/40 [00:00<?, ?it/s]

Evaluate:   0%|          | 0/4 [00:00<?, ?it/s]

,run_name,seeds,validation_accuracy_mean,validation_accuracy_std
0,neural_baseline_lambda_s_0,5,0.2604,0.005079
1,neural_claq_lambda_s_0_4,5,0.2590,0.005958


In [9]:
from scipy.stats import t

curve_frames = []
detail_frames = []
query_frames = []

for run in trained_runs:
    curve, detail, queries = rollout_neural_claq(
        actor=run["actor"],
        classifier=run["classifier"],
        loader=test_loader,
        run_name=run["run_name"],
        seed=run["seed"],
    )
    curve_frames.append(curve)
    detail_frames.append(detail)
    query_frames.append(queries)

questioner_curve_by_seed = pd.concat(curve_frames, ignore_index=True)
questioner_detail_by_seed = pd.concat(detail_frames, ignore_index=True)
questioner_queries_by_seed = pd.concat(query_frames, ignore_index=True)
stopping_summary_by_seed = pd.concat(
    [summarize_confidence_stopping(detail) for detail in detail_frames],
    ignore_index=True,
)
query_usage_by_seed = summarize_query_usage(questioner_queries_by_seed)

questioner_curve = (
    questioner_curve_by_seed.groupby(["run_name", "budget"])
    .agg(
        seeds=("seed", "nunique"),
        accuracy=("accuracy", "mean"),
        accuracy_std=("accuracy", "std"),
        macro_f1=("macro_f1", "mean"),
        macro_f1_std=("macro_f1", "std"),
        mean_confidence=("mean_confidence", "mean"),
    )
    .reset_index()
)
questioner_curve["accuracy_ci95"] = (
    t.ppf(0.975, questioner_curve["seeds"] - 1)
    * questioner_curve["accuracy_std"]
    / np.sqrt(questioner_curve["seeds"])
)

stopping_summary = (
    stopping_summary_by_seed.groupby(["run_name", "confidence_threshold"])
    .agg(
        seeds=("seed", "nunique"),
        coverage=("coverage", "mean"),
        coverage_std=("coverage", "std"),
        mean_questions_when_reached=("mean_questions_when_reached", "mean"),
        accuracy_when_reached=("accuracy_when_reached", "mean"),
    )
    .reset_index()
)
# Include zero counts when a concept is not selected by a seed. Grouping only
# observed rows would overstate mean usage and leave some standard deviations undefined.
query_usage_index = pd.MultiIndex.from_product(
    [
        sorted(query_usage_by_seed["run_name"].unique()),
        SEEDS,
        range(1, MAX_QUESTIONS + 1),
        concept_names,
    ],
    names=["run_name", "seed", "budget", "query_concept"],
)
query_usage_complete = (
    query_usage_by_seed.set_index(["run_name", "seed", "budget", "query_concept"])
    .reindex(query_usage_index, fill_value=0)
    .reset_index()
)
query_usage = (
    query_usage_complete.groupby(["run_name", "budget", "query_concept"])
    .agg(count=("count", "mean"), count_std=("count", "std"))
    .reset_index()
    .sort_values(["run_name", "budget", "count"], ascending=[True, True, False])
)

sensitive_concept_names = set(
    concepts_df.loc[sensitive_indices.cpu().numpy(), "concept"].tolist()
)
sensitive_query_summary = (
    query_usage_complete.assign(
        is_sensitive=lambda frame: frame["query_concept"].isin(sensitive_concept_names),
        sensitive_count=lambda frame: frame["count"] * frame["is_sensitive"],
    )
    .groupby(["run_name", "seed"], as_index=False)
    .agg(sensitive_queries=("sensitive_count", "sum"))
)
sensitive_query_summary["sensitive_query_rate"] = (
    sensitive_query_summary["sensitive_queries"]
    / (len(test_loader.dataset) * MAX_QUESTIONS)
)
sensitive_query_summary_aggregate = (
    sensitive_query_summary.groupby("run_name", as_index=False)
    .agg(
        seeds=("seed", "nunique"),
        sensitive_query_rate=("sensitive_query_rate", "mean"),
        sensitive_query_rate_std=("sensitive_query_rate", "std"),
    )
)

# Per-example qualitative analyses use one declared representative seed; all
# quantitative summaries above include every seed.
QUALITATIVE_SEED = SEEDS[0]
questioner_detail = questioner_detail_by_seed.loc[
    questioner_detail_by_seed["seed"].eq(QUALITATIVE_SEED)
].copy()
questioner_queries = questioner_queries_by_seed.loc[
    questioner_queries_by_seed["seed"].eq(QUALITATIVE_SEED)
].copy()

questioner_curve.pivot(index="budget", columns="run_name", values="accuracy").head(MAX_QUESTIONS + 1)


Rollout neural_baseline_lambda_s_0:   0%|          | 0/8 [00:00<?, ?it/s]

Rollout neural_claq_lambda_s_0_4:   0%|          | 0/8 [00:00<?, ?it/s]

Rollout neural_baseline_lambda_s_0:   0%|          | 0/8 [00:00<?, ?it/s]

Rollout neural_claq_lambda_s_0_4:   0%|          | 0/8 [00:00<?, ?it/s]

Rollout neural_baseline_lambda_s_0:   0%|          | 0/8 [00:00<?, ?it/s]

...

Rollout neural_baseline_lambda_s_0:   0%|          | 0/8 [00:00<?, ?it/s]

Rollout neural_claq_lambda_s_0_4:   0%|          | 0/8 [00:00<?, ?it/s]

Rollout neural_baseline_lambda_s_0:   0%|          | 0/8 [00:00<?, ?it/s]

Rollout neural_claq_lambda_s_0_4:   0%|          | 0/8 [00:00<?, ?it/s]

run_name,neural_baseline_lambda_s_0,neural_claq_lambda_s_0_4
budget,,
0,0.0355,0.0358
1,0.0655,0.0679
2,0.1206,0.1186
3,0.1937,0.1813
4,0.2783,0.2552
5,0.3274,0.3046
6,0.3690,0.3261
7,0.3937,0.3419
8,0.4141,0.3619


In [10]:
def format_query_path(query_df, run_name, row_idx, max_budget=10):
    path = query_df.loc[
        query_df["run_name"].eq(run_name)
        & query_df["row_idx"].eq(row_idx)
        & query_df["budget"].le(max_budget)
    ].sort_values("budget")
    return " -> ".join(path["query_concept"].tolist())


def prediction_snapshot(detail_df, run_name, row_idx, budget):
    row = detail_df.loc[
        detail_df["run_name"].eq(run_name)
        & detail_df["row_idx"].eq(row_idx)
        & detail_df["budget"].eq(budget)
    ].iloc[0]
    return {
        "prediction": profession_id_to_name[int(row["prediction"])],
        "confidence": round(float(row["confidence"]), 3),
        "correct": bool(row["correct"]),
    }


def choose_comparison_rows(detail_df, max_rows=28):
    final_budget = int(detail_df["budget"].max())
    final_predictions = detail_df.loc[detail_df["budget"].eq(final_budget)].pivot(
        index="row_idx",
        columns="run_name",
        values="prediction",
    )
    final_correct = detail_df.loc[detail_df["budget"].eq(final_budget)].pivot(
        index="row_idx",
        columns="run_name",
        values="correct",
    )

    differing_rows = final_predictions.nunique(axis=1).gt(1) | final_correct.nunique(axis=1).gt(1)
    ranked = final_predictions.index[differing_rows].tolist()

    confidence_gap_rows = (
        detail_df.sort_values(["row_idx", "budget"])
        .drop_duplicates(["run_name", "row_idx"], keep="last")
        .groupby("row_idx")["confidence"]
        .std()
        .sort_values(ascending=False)
        .index.tolist()
    )
    ranked.extend([row_idx for row_idx in confidence_gap_rows if row_idx not in ranked])

    professions = {
        int(row_idx): split_frames["test"].iloc[int(row_idx)]["profession_name"]
        for row_idx in ranked
    }

    selected = []
    used_professions = set()

    # First pass: prefer one example per profession.
    for row_idx in ranked:
        profession = professions[int(row_idx)]
        if profession in used_professions:
            continue
        selected.append(row_idx)
        used_professions.add(profession)
        if len(selected) >= max_rows:
            return selected

    # Fallback: fill remaining rows if max_rows is larger than available professions.
    selected.extend([row_idx for row_idx in ranked if row_idx not in selected])
    return selected[:max_rows]


def build_sample_comparison(detail_df, query_df, budgets=(5, 10, 20), max_query_budget=10, max_rows=28):
    rows = []
    run_names = sorted(detail_df["run_name"].unique())
    comparison_rows = choose_comparison_rows(detail_df, max_rows=max_rows)

    for row_idx in comparison_rows:
        test_row = split_frames["test"].iloc[int(row_idx)]
        row = {
            "row_idx": int(row_idx),
            "sample_row": int(test_row["sample_row"]),
            "true_profession": test_row["profession_name"],
            "biography": test_row[TEXT_COLUMN][:500],
        }
        for run_name in run_names:
            row[f"{run_name}_first_{max_query_budget}_questions"] = format_query_path(
                query_df,
                run_name,
                row_idx,
                max_budget=max_query_budget,
            )
            for budget in budgets:
                snapshot = prediction_snapshot(detail_df, run_name, row_idx, budget)
                prefix = f"{run_name}_budget_{budget}"
                row[f"{prefix}_prediction"] = snapshot["prediction"]
                row[f"{prefix}_confidence"] = snapshot["confidence"]
                row[f"{prefix}_correct"] = snapshot["correct"]
        rows.append(row)

    return pd.DataFrame(rows)


sample_comparison = build_sample_comparison(questioner_detail, questioner_queries)


In [11]:
pd.set_option("display.max_colwidth", 600)

overview_cols = ["row_idx", "sample_row", "true_profession", "biography"]
display(sample_comparison[overview_cols])

question_cols = [col for col in sample_comparison.columns if "questions" in col]
display(sample_comparison[["row_idx", *question_cols]])

prediction_cols = [
    col for col in sample_comparison.columns
    if "budget" in col and ("prediction" in col or "confidence" in col or "correct" in col)
]
display(sample_comparison[["row_idx", *prediction_cols]])


,row_idx,sample_row,true_profession,biography
0,0,0,accountant,She is primarily responsible for trust and estate tax returns. She is also very proficient at the preparation of Form 8824 Like-Kind Exchanges and structuring real estate transactions. [More details]
1,4,4,physician,"In the ensuing years, she held a variety of positions in the health care industry acquiring a diverse toolbox of skills. She joined KMC University in 2016."
2,6,6,paralegal,"She joined the firm in March of 2018. Her role is to assist the firm’s attorneys in making sure deadlines are met, clients are up to date on case process, and ensure each day runs as smooth as possible. Tatiana graduated from Sheldon High School in 2012 and studied at Sacramento City and San Diego City colleges. She received her Paralegal Certificate from California State University East Bay in June 2017."
3,11,11,chiropractor,"He has been a consultant to insurance companies, worker's compensation and formerly president of the Ontario and Canadian Chiropractic Associations. He chaired and edited the C.C.A.'s Clinical Practice Guidelines (Glenerin) Document."
4,14,14,attorney,"Her practice focuses primarily on business immigration, including working with clients in a variety of industries to prepare and file non-immigrant and immigrant visas. She has extensive experience with L-1 visas, including blanket and individual L-1s, as well as new office L petitions. She is a 2004 graduate of American University Washington College of Law and earned her undergraduate degree at Gonzaga University in 2001. She currently serves on the Board of AILA’s Northern California Chapter a"
5,32,32,teacher,"While teaching, she has researched her own teaching philosophies and strategies, and has paid close attention to how children share their understandings. She also focuses on the role of the arts in learning, and has written some books about her experiences as a teacher-researcher and her findings. She received her Ed.D. in education from Boston University in 1981 and is a member of the Brookline Teacher Research Seminar."
6,39,39,personal_trainer,"Her interest in fitness started young with sports but after discovering she had a congenital heart defect at age 19, her interest turned into passion. It took multiple surgeries and almost 2 years to fix this condition, and after attending cardiac rehab, wellness became her main focus. After finishing a degree in Psychology it was followed up by a diploma in Personal Training, to make wellness not only a focus in her own life but in the life of others."
7,45,45,pastor,"Along with The Body Battle message, she is passionate about empowering others to live by faith and fulfill their destiny through a close relationship with God."
8,71,71,painter,"He begins by sketching and framing the object he wants to create and creating a color study. Wine meticulously lays down tape and adjusts, cuts, and moves the tape to create clean shapes. He then paints and peels the tape up for a “slow, wonderful, pain-staking reveal.” Wine is a College of Charleston alumnus and works at Artist and Craftsman Supply."
9,74,74,interior_designer,"Indefatigable importer of the anglo-saxon culture, absorbed during his professional career, he combines since 1997 turnkey design and all round consulting services. His interior projects go from private to retail."


,row_idx,neural_baseline_lambda_s_0_first_10_questions,neural_claq_lambda_s_0_4_first_10_questions
0,0,health_or_wellness_work -> creative_or_artistic_work -> female_pronouns -> legal_or_regulatory_work -> clinical_care -> design_work -> visual_art_practice -> literary_writing -> film_or_video_production -> live_music_or_recording,creative_or_artistic_work -> health_or_wellness_work -> legal_or_regulatory_work -> technical_or_engineering_work -> design_work -> clinical_care -> visual_art_practice -> live_music_or_recording -> mind_body_practice -> film_or_video_production
1,4,health_or_wellness_work -> nursing_care -> female_pronouns -> clinical_care -> creative_or_artistic_work -> dental_or_oral_health -> legal_or_regulatory_work -> design_work -> male_pronouns -> nutrition_or_diet_guidance,creative_or_artistic_work -> health_or_wellness_work -> mind_body_practice -> clinical_care -> nutrition_or_diet_guidance -> legal_or_regulatory_work -> dental_or_oral_health -> nursing_care -> technical_or_engineering_work -> design_work
2,6,health_or_wellness_work -> creative_or_artistic_work -> female_pronouns -> legal_or_regulatory_work -> clinical_care -> design_work -> visual_art_practice -> literary_writing -> film_or_video_production -> live_music_or_recording,creative_or_artistic_work -> health_or_wellness_work -> legal_or_regulatory_work -> technical_or_engineering_work -> design_work -> clinical_care -> visual_art_practice -> live_music_or_recording -> mind_body_practice -> film_or_video_production
3,11,health_or_wellness_work -> nursing_care -> female_pronouns -> clinical_care -> dental_or_oral_health -> creative_or_artistic_work -> male_pronouns -> nutrition_or_diet_guidance -> legal_or_regulatory_work -> mind_body_practice,creative_or_artistic_work -> health_or_wellness_work -> mind_body_practice -> clinical_care -> dental_or_oral_health -> nursing_care -> nutrition_or_diet_guidance -> legal_or_regulatory_work -> technical_or_engineering_work -> education_or_academic_work
4,14,health_or_wellness_work -> creative_or_artistic_work -> female_pronouns -> legal_or_regulatory_work -> clinical_care -> design_work -> visual_art_practice -> literary_writing -> film_or_video_production -> live_music_or_recording,creative_or_artistic_work -> health_or_wellness_work -> legal_or_regulatory_work -> technical_or_engineering_work -> design_work -> clinical_care -> visual_art_practice -> live_music_or_recording -> mind_body_practice -> film_or_video_production
5,32,health_or_wellness_work -> creative_or_artistic_work -> female_pronouns -> legal_or_regulatory_work -> design_work -> clinical_care -> visual_art_practice -> literary_writing -> film_or_video_production -> live_music_or_recording,creative_or_artistic_work -> health_or_wellness_work -> legal_or_regulatory_work -> technical_or_engineering_work -> design_work -> clinical_care -> visual_art_practice -> live_music_or_recording -> mind_body_practice -> film_or_video_production
6,39,health_or_wellness_work -> nursing_care -> female_pronouns -> clinical_care -> creative_or_artistic_work -> dental_or_oral_health -> legal_or_regulatory_work -> design_work -> male_pronouns -> nutrition_or_diet_guidance,creative_or_artistic_work -> health_or_wellness_work -> mind_body_practice -> clinical_care -> nutrition_or_diet_guidance -> legal_or_regulatory_work -> dental_or_oral_health -> nursing_care -> technical_or_engineering_work -> design_work
7,45,health_or_wellness_work -> creative_or_artistic_work -> female_pronouns -> legal_or_regulatory_work -> design_work -> clinical_care -> visual_art_practice -> literary_writing -> film_or_video_production -> live_music_or_recording,creative_or_artistic_work -> health_or_wellness_work -> legal_or_regulatory_work -> technical_or_engineering_work -> design_work -> clinical_care -> visual_art_practice -> live_music_or_recording -> mind_body_practice -> film_or_video_production
8,71,health_or_wellness_work -> creative_or_artistic_work -> visual_art_practice -> l

,row_idx,neural_baseline_lambda_s_0_budget_5_prediction,neural_baseline_lambda_s_0_budget_5_confidence,neural_baseline_lambda_s_0_budget_5_correct,neural_baseline_lambda_s_0_budget_10_prediction,neural_baseline_lambda_s_0_budget_10_confidence,neural_baseline_lambda_s_0_budget_10_correct,neural_baseline_lambda_s_0_budget_20_prediction,neural_baseline_lambda_s_0_budget_20_confidence,neural_baseline_lambda_s_0_budget_20_correct,neural_claq_lambda_s_0_4_budget_5_prediction,neural_claq_lambda_s_0_4_budget_5_confidence,neural_claq_lambda_s_0_4_budget_5_correct,neural_claq_lambda_s_0_4_budget_10_prediction,neural_claq_lambda_s_0_4_budget_10_confidence,neural_claq_lambda_s_0_4_budget_10_correct,neural_claq_lambda_s_0_4_budget_20_prediction,neural_claq_lambda_s_0_4_budget_20_confidence,neural_claq_lambda_s_0_4_budget_20_correct
0,0,paralegal,0.687,False,paralegal,0.677,False,paralegal,0.584,False,attorney,0.599,False,attorney,0.595,False,attorney,0.532,False
1,4,dietitian,0.320,False,yoga_teacher,0.456,False,dietitian,0.195,False,personal_trainer,0.311,False,personal_trainer,0.288,False,personal_trainer,0.263,False
2,6,paralegal,0.687,True,paralegal,0.677,True,paralegal,0.584,True,attorney,0.599,False,attorney,0.595,False,attorney,0.486,False
3,11,surgeon,0.318,False,surgeon,0.341,False,psychologist,0.250,False,nurse,0.215,False,psychologist,0.339,False,chiropractor,0.297,True
4,14,paralegal,0.687,False,paralegal,0.677,False,paralegal,0.793,False,attorney,0.599,True,attorney,0.595,True,attorney,0.486,True
5,32,model,0.165,False,model,0.187,False,model,0.218,False,pastor,0.110,False,accountant,0.164,False,accountant,0.214,False
6,39,dietitian,0.320,False,yoga_teacher,0.456,False,dietitian,0.195,False,personal_trainer,0.311,True,personal_trainer,0.288,True,personal_trainer,0.263,True
7,45,model,0.165,False,model,0.187,False,model,0.218,False,pastor,0.110,True,accountant,0.164,False,accountant,0.221,False
8,71,composer,0.172,False,composer,0.173,False,rapper,0.213,False,filmmaker,0.168,False,model,0.141,False,teacher,0.122,False
9,74,interior_designer,0.788,True,interior_designer,0.552,True,architect,0.660,False,interior_designer,0.850,True,interior_designer,0.840,True,interior_designer,0.648,True


In [12]:
best_budget_rows = (
    questioner_curve.sort_values(["run_name", "accuracy"], ascending=[True, False])
    .groupby("run_name")
    .head(1)
    .reset_index(drop=True)
)

{
    "best_budget_by_run": best_budget_rows.to_dict(orient="records"),
    "sensitive_query_usage": sensitive_query_summary_aggregate.to_dict(orient="records"),
    "confidence_stopping": stopping_summary.to_dict(orient="records"),
    "top_queries_first_5_budgets": query_usage.loc[query_usage["budget"].le(5)].groupby(["run_name", "budget"]).head(5).to_dict(orient="records"),
}


{'best_budget_by_run': [{'run_name': 'neural_baseline_lambda_s_0',
   'budget': 20,
   'seeds': 5,
   'accuracy': 0.5379,
   'accuracy_std': 0.01233085560697236,
   'macro_f1': 0.5335891498010933,
   'macro_f1_std': 0.019060132181094994,
   'mean_confidence': 0.5190499365329743,
   'accuracy_ci95': 0.015310779474226602},
  {'run_name': 'neural_claq_lambda_s_0_4',
   'budget': 20,
   'seeds': 5,
   'accuracy': 0.4981,
   'accuracy_std': 0.026959228475607368,
   'macro_f1': 0.48988675058870673,
   'macro_f1_std': 0.035502833592447755,
   'mean_confidence': 0.46162163019180297,
   'accuracy_ci95': 0.03347430341751141}],
 'sensitive_query_usage': [{'run_name': 'neural_baseline_lambda_s_0',
   'seeds': 5,
   'sensitive_query_rate': 0.11976499999999998,
   'sensitive_query_rate_std': 0.009707996059949745},
  {'run_name': 'neural_claq_lambda_s_0_4',
   'seeds': 5,
   'sensitive_query_rate': 0.034960000000000005,
   'sensitive_query_rate_std': 0.01287932694281809}],
 'confidence_stopping': [{'

In [13]:
experiment_name = EXPERIMENT
curve_path = runs_dir / f"{experiment_name}_curve.csv"
curve_by_seed_path = runs_dir / f"{experiment_name}_curve_by_seed.csv"
detail_path = runs_dir / f"{experiment_name}_detail_by_seed.csv"
stopping_path = runs_dir / f"{experiment_name}_stopping.csv"
stopping_by_seed_path = runs_dir / f"{experiment_name}_stopping_by_seed.csv"
queries_path = runs_dir / f"{experiment_name}_queries_by_seed.csv"
history_path = runs_dir / f"{experiment_name}_history.csv"
query_usage_path = runs_dir / f"{experiment_name}_query_usage.csv"
query_usage_by_seed_path = runs_dir / f"{experiment_name}_query_usage_by_seed.csv"
sensitive_query_summary_path = runs_dir / f"{experiment_name}_sensitive_query_summary.csv"
sensitive_query_summary_by_seed_path = runs_dir / f"{experiment_name}_sensitive_query_summary_by_seed.csv"
sample_comparison_path = runs_dir / f"{experiment_name}_sample_comparison.csv"
summary_path = runs_dir / f"{experiment_name}_summary.json"
checkpoint_manifest_path = models_dir / f"{experiment_name}_checkpoints.json"

questioner_curve.to_csv(curve_path, index=False, lineterminator="\n")
questioner_curve_by_seed.to_csv(curve_by_seed_path, index=False, lineterminator="\n")
questioner_detail_by_seed.to_csv(detail_path, index=False, lineterminator="\n")
stopping_summary.to_csv(stopping_path, index=False, lineterminator="\n")
stopping_summary_by_seed.to_csv(stopping_by_seed_path, index=False, lineterminator="\n")
questioner_queries_by_seed.to_csv(queries_path, index=False, lineterminator="\n")
history_df.to_csv(history_path, index=False, lineterminator="\n")
query_usage.to_csv(query_usage_path, index=False, lineterminator="\n")
query_usage_by_seed.to_csv(query_usage_by_seed_path, index=False, lineterminator="\n")
sensitive_query_summary_aggregate.to_csv(
    sensitive_query_summary_path, index=False, lineterminator="\n"
)
sensitive_query_summary.to_csv(
    sensitive_query_summary_by_seed_path, index=False, lineterminator="\n"
)
sample_comparison.to_csv(sample_comparison_path, index=False, lineterminator="\n")

checkpoint_manifest = {
    "format_version": 1,
    "experiment_name": experiment_name,
    "concept_names": concept_names,
    "profession_id_to_name": profession_id_to_name,
    "sensitive_conditioning": "conditional_y",
    "runs": {
        f"{run['run_name']}_seed_{run['seed']}": {
            "run_name": run["run_name"],
            "seed": run["seed"],
            "lambda_s": run["lambda_s"],
            "best_validation_accuracy": run["best"]["validation_accuracy"],
            "best_epoch": run["best"]["epoch"],
            "checkpoint": str(run["checkpoint_path"].relative_to(repo_root)),
        }
        for run in trained_runs
    },
}
with open(checkpoint_manifest_path, "w", encoding="utf-8") as handle:
    json.dump(checkpoint_manifest, handle, indent=2)

summary = {
    "experiment_name": experiment_name,
    "target": "original_28_way_profession",
    "answer_source": "hard Concept-QA answers converted to -1/+1",
    "training": "neural actor/classifier trained end-to-end on sampled partial histories",
    "decision_threshold": decision_threshold,
    "sensitive_target": "dataset gender label, 1=female and 0=male",
    "sensitive_conditioning": "encoded knowledge state and one-hot profession label",
    "max_questions": MAX_QUESTIONS,
    "num_epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "seeds": list(SEEDS),
    "qualitative_seed": QUALITATIVE_SEED,
    "history_config": {
        "min_history": history_config.min_history,
        "max_history": history_config.max_history,
        "non_sensitive_only": history_config.non_sensitive_only,
    },
    "num_concepts": len(concept_names),
    "num_professions": num_professions,
    "best_budget_by_run": best_budget_rows.to_dict(orient="records"),
    "sensitive_query_usage": sensitive_query_summary_aggregate.to_dict(orient="records"),
    "outputs": {
        "curve": str(curve_path.relative_to(repo_root)),
        "curve_by_seed": str(curve_by_seed_path.relative_to(repo_root)),
        "detail_by_seed": str(detail_path.relative_to(repo_root)),
        "stopping": str(stopping_path.relative_to(repo_root)),
        "stopping_by_seed": str(stopping_by_seed_path.relative_to(repo_root)),
        "queries_by_seed": str(queries_path.relative_to(repo_root)),
        "history": str(history_path.relative_to(repo_root)),
        "query_usage": str(query_usage_path.relative_to(repo_root)),
        "query_usage_by_seed": str(query_usage_by_seed_path.relative_to(repo_root)),
        "sensitive_query_summary": str(sensitive_query_summary_path.relative_to(repo_root)),
        "sensitive_query_summary_by_seed": str(sensitive_query_summary_by_seed_path.relative_to(repo_root)),
        "sample_comparison": str(sample_comparison_path.relative_to(repo_root)),
        "checkpoint_manifest": str(checkpoint_manifest_path.relative_to(repo_root)),
        "checkpoints": [
            str(run["checkpoint_path"].relative_to(repo_root)) for run in trained_runs
        ],
    },
}
with summary_path.open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, indent=2)

summary["outputs"] | {"summary": str(summary_path.relative_to(repo_root))}


{'curve': 'artifacts/runs/bias_in_bios_leakage_curve.csv',
 'curve_by_seed': 'artifacts/runs/bias_in_bios_leakage_curve_by_seed.csv',
 'detail_by_seed': 'artifacts/runs/bias_in_bios_leakage_detail_by_seed.csv',
 'stopping': 'artifacts/runs/bias_in_bios_leakage_stopping.csv',
 'stopping_by_seed': 'artifacts/runs/bias_in_bios_leakage_stopping_by_seed.csv',
 'queries_by_seed': 'artifacts/runs/bias_in_bios_leakage_queries_by_seed.csv',
 'history': 'artifacts/runs/bias_in_bios_leakage_history.csv',
 'query_usage': 'artifacts/runs/bias_in_bios_leakage_query_usage.csv',
 'query_usage_by_seed': 'artifacts/runs/bias_in_bios_leakage_query_usage_by_seed.csv',
 'sensitive_query_summary': 'artifacts/runs/bias_in_bios_leakage_sensitive_query_summary.csv',
 'sensitive_query_summary_by_seed': 'artifacts/runs/bias_in_bios_leakage_sensitive_query_summary_by_seed.csv',
 'sample_comparison': 'artifacts/runs/bias_in_bios_leakage_sample_comparison.csv',
 'checkpoint_manifest': 'artifacts/models/bias_in_bios

## Qualitative case cards

We render a few full biographies as annotated case cards. Red marks the gender / pronoun cues that the baseline questioner opens on, and green marks profession evidence in the text. The examples are chosen to be representative rather than cherry-picked: in most cards both questioners reach the correct profession. The point is the acquisition behavior, not accuracy: the baseline spends its first query on `female_pronouns` for every sample, while CLAQ begins from profession-relevant concepts and reaches the same answer without the gender-coded query. The figure is written to `artifacts/figures/bias_in_bios_case_cards.svg`.

In [14]:
from claq.analysis.bias_in_bios_cards import select_card_examples, plot_bias_in_bios_cards

figures_dir = repo_root / "artifacts" / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)

run_names = [run["run_name"] for run in trained_runs]
baseline_run = next(name for name in run_names if "baseline" in name)
claq_run = next(name for name in run_names if name != baseline_run)

card_examples = select_card_examples(
    questioner_detail,
    questioner_queries,
    split_frames["test"],
    baseline_run=baseline_run,
    claq_run=claq_run,
    n=3,
)
for ex in card_examples:
    print(
        ex["row_idx"], ex["true_profession"], f"({ex['gender']})",
        "| baseline:", ex["baseline"]["pred"], ex["baseline"]["correct"],
        "| CLAQ:", ex["claq"]["pred"], ex["claq"]["correct"],
    )

cards_path = plot_bias_in_bios_cards(card_examples, figures_dir / "bias_in_bios_case_cards.svg")
cards_path

ValueError: no examples to plot